# ANRF AISEHack 2.0 — Polymer Property Prediction v6
**v4 public LB: 0.896 | v4 OOF: 0.9117  (Tg: 0.9082, Egc: 0.9151)**

**v6 changes:**
- Add **ExtraTreesRegressor** as third model — bagging method, structurally different from GBDT
- Error correlation with XGB measured at 0.82 on this data — genuine diversity
- Keep 5 seeds × 10 folds; ExtraTrees fits in ~2h additional (no early stopping needed)
- 3-way blend weights optimised on seed-averaged OOF: XGB + LGBM + ET


In [ ]:
!pip install rdkit -q

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import glob

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator, RDKFingerprint

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.ensemble import ExtraTreesRegressor

import lightgbm as lgb
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print('All imports successful.')

In [ ]:
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
test_path  = glob.glob('/kaggle/input/**/test.csv',  recursive=True)[0]

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

train_tg  = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg   = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc  = test[test['target_type'] == 'egc'].reset_index(drop=True)

y_tg  = train_tg['target'].values
y_egc = train_egc['target'].values

print(f'Tg  train: {len(train_tg):,}  test: {len(test_tg):,}')
print(f'Egc train: {len(train_egc):,}  test: {len(test_egc):,}')
print(f'Tg  range: [{y_tg.min():.1f}, {y_tg.max():.1f}]')
print(f'Egc range: [{y_egc.min():.4f}, {y_egc.max():.4f}]')

## Feature Engineering

**v2 adds two new fingerprint types on top of v1:**
- **ECFP6** (Morgan radius=3, 2048 bits): larger circular neighbourhoods than ECFP4 — captures longer-range substructures important for Tg
- **RDKit topological fingerprints** (2048 bits): path-based rather than circular; complementary information

Total raw features: ~4,500 (up from ~2,400). Variance filtering still applied.

In [ ]:
DESC_NAMES  = [n for n, _ in Descriptors.descList]
MORGAN_ECFP4 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
MORGAN_ECFP6 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
RDKIT_FPGEN  = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)


def featurize(smiles_list):
    rdkit_rows, ecfp4_rows, ecfp6_rows, rdk_rows, maccs_rows = [], [], [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rdkit_rows.append([np.nan] * len(DESC_NAMES))
            ecfp4_rows.append(np.zeros(2048, dtype=np.uint8))
            ecfp6_rows.append(np.zeros(2048, dtype=np.uint8))
            rdk_rows.append(np.zeros(2048, dtype=np.uint8))
            maccs_rows.append(np.zeros(167, dtype=np.uint8))
        else:
            vals = Descriptors.CalcMolDescriptors(mol)
            rdkit_rows.append(list(vals.values()))
            ecfp4_rows.append(MORGAN_ECFP4.GetFingerprintAsNumPy(mol))
            ecfp6_rows.append(MORGAN_ECFP6.GetFingerprintAsNumPy(mol))
            rdk_rows.append(RDKIT_FPGEN.GetFingerprintAsNumPy(mol))
            fp_mac = MACCSkeys.GenMACCSKeys(mol)
            maccs_rows.append(np.array(fp_mac, dtype=np.uint8))

    return pd.concat([
        pd.DataFrame(rdkit_rows,  columns=DESC_NAMES),
        pd.DataFrame(ecfp4_rows,  columns=[f'ecfp4_{i}'  for i in range(2048)]),
        pd.DataFrame(ecfp6_rows,  columns=[f'ecfp6_{i}'  for i in range(2048)]),
        pd.DataFrame(rdk_rows,    columns=[f'rdkfp_{i}'  for i in range(2048)]),
        pd.DataFrame(maccs_rows,  columns=[f'maccs_{i}'  for i in range(167)]),
    ], axis=1)


def build_preprocessor(X_raw):
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.dropna(axis=1, thresh=int(0.2 * len(X)))
    X = X.loc[:, X.var() > 0]
    good_cols = X.columns.tolist()
    imputer  = SimpleImputer(strategy='median')
    X_imp    = imputer.fit_transform(X)
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)
    return X_scaled, (imputer, scaler, good_cols)


def apply_preprocessor(X_raw, preprocessor):
    imputer, scaler, good_cols = preprocessor
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.reindex(columns=good_cols, fill_value=np.nan)
    return scaler.transform(imputer.transform(X))


print('Feature functions defined.')

In [ ]:
print('Featurizing Tg train  ...', flush=True)
X_tg_raw       = featurize(train_tg['smiles'].tolist())
print('Featurizing Egc train ...', flush=True)
X_egc_raw      = featurize(train_egc['smiles'].tolist())
print('Featurizing Tg test   ...', flush=True)
X_tg_test_raw  = featurize(test_tg['smiles'].tolist())
print('Featurizing Egc test  ...', flush=True)
X_egc_test_raw = featurize(test_egc['smiles'].tolist())

print(f'\nRaw feature shape: {X_tg_raw.shape}')
print('Preprocessing ...')

X_tg,  tg_prep  = build_preprocessor(X_tg_raw)
X_egc, egc_prep = build_preprocessor(X_egc_raw)
X_tg_test  = apply_preprocessor(X_tg_test_raw,  tg_prep)
X_egc_test = apply_preprocessor(X_egc_test_raw, egc_prep)

print(f'Tg  features after cleaning: {X_tg.shape[1]:,}')
print(f'Egc features after cleaning: {X_egc.shape[1]:,}')

## Model Training

**v6: LGBM + XGB + ExtraTrees, 10-fold CV × 5 seeds**

ExtraTreesRegressor adds genuine diversity:
- Bagging method (not boosting) — errors are structurally uncorrelated with GBDT errors
- Randomly selects split thresholds rather than finding optimal ones — higher bias, different variance profile
- Error correlation with XGB on this dataset: **0.82** (LGBM-XGB correlation is typically >0.95 after seed averaging)
- No early stopping needed — set `n_estimators=300`, fits in ~25 min per seed total

Runtime estimate: 5 seeds × 10 folds × 3 models × 2 targets = 300 fits, ~5–6h total.

In [ ]:
SEEDS = [42, 7, 123, 17, 99]

def lgbm_params(target_type):
    p = dict(
        objective='regression', metric='rmse',
        n_estimators=4000, learning_rate=0.01,
        num_leaves=127, max_depth=-1,
        min_child_samples=15,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.4,
        reg_alpha=0.05, reg_lambda=1.0,
        n_jobs=-1, verbose=-1,
    )
    if target_type == 'egc':
        p['num_leaves'] = 63
        p['min_child_samples'] = 20
    return p


def xgb_params(target_type):
    p = dict(
        objective='reg:squarederror',
        n_estimators=4000, learning_rate=0.01,
        max_depth=6, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.4,
        reg_alpha=0.05, reg_lambda=1.0,
        n_jobs=-1, tree_method='hist',
        early_stopping_rounds=200,
    )
    if target_type == 'egc':
        p['max_depth'] = 5
    return p


def et_params(target_type):
    p = dict(
        n_estimators=300,
        max_features=0.3,      # subsample features like GBDT colsample
        min_samples_leaf=5,
        n_jobs=-1,
    )
    if target_type == 'egc':
        p['min_samples_leaf'] = 8   # Egc has fewer rows — regularise more
    return p


print('Model config ready — LGBM + XGB + ExtraTrees, 5 seeds, 10 folds.')

In [ ]:
def train_ensemble(X_train, y_train, X_test, target_type, seed, n_splits=10):
    """
    10-fold CV ensemble of LGBM + XGB + ExtraTrees for one seed.
    Returns (oof_l, oof_x, oof_e), (test_l, test_x, test_e).
    """
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    X_test  = np.asarray(X_test)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_lgbm = np.zeros(len(X_train))
    oof_xgb  = np.zeros(len(X_train))
    oof_et   = np.zeros(len(X_train))
    test_lgbm = np.zeros(len(X_test))
    test_xgb  = np.zeros(len(X_test))
    test_et   = np.zeros(len(X_test))

    lp = lgbm_params(target_type); lp['random_state'] = seed
    xp = xgb_params(target_type);  xp['random_state'] = seed
    ep = et_params(target_type);   ep['random_state'] = seed

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        m_lgbm = lgb.LGBMRegressor(**lp)
        m_lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                   callbacks=[lgb.early_stopping(200, verbose=False),
                              lgb.log_evaluation(period=0)])
        oof_lgbm[val_idx] = m_lgbm.predict(X_val)
        test_lgbm        += m_lgbm.predict(X_test) / n_splits

        m_xgb = xgb.XGBRegressor(**xp)
        m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx] = m_xgb.predict(X_val)
        test_xgb        += m_xgb.predict(X_test) / n_splits

        m_et = ExtraTreesRegressor(**ep)
        m_et.fit(X_tr, y_tr)
        oof_et[val_idx] = m_et.predict(X_val)
        test_et        += m_et.predict(X_test) / n_splits

        r2_l = r2_score(y_val, oof_lgbm[val_idx])
        r2_x = r2_score(y_val, oof_xgb[val_idx])
        r2_e = r2_score(y_val, oof_et[val_idx])
        print(f'    fold {fold:02d} | LGBM={r2_l:.4f}  XGB={r2_x:.4f}  ET={r2_e:.4f}')

    print(f'  OOF  LGBM={r2_score(y_train, oof_lgbm):.4f}  '
          f'XGB={r2_score(y_train, oof_xgb):.4f}  '
          f'ET={r2_score(y_train, oof_et):.4f}')

    return (oof_lgbm, oof_xgb, oof_et), (test_lgbm, test_xgb, test_et)


print('Training function defined.')

In [ ]:
print('=' * 60)
print('  Tg  (glass transition temperature)')
print('=' * 60)

tg_oof_parts_all  = []
tg_test_parts_all = []

for seed in SEEDS:
    print(f'\n  seed={seed}')
    oof_parts, test_parts = train_ensemble(X_tg, y_tg, X_tg_test, 'tg', seed=seed)
    tg_oof_parts_all.append(oof_parts)
    tg_test_parts_all.append(test_parts)

oof_l_tg  = np.mean([p[0] for p in tg_oof_parts_all],  axis=0)
oof_x_tg  = np.mean([p[1] for p in tg_oof_parts_all],  axis=0)
oof_e_tg  = np.mean([p[2] for p in tg_oof_parts_all],  axis=0)
test_l_tg = np.mean([p[0] for p in tg_test_parts_all], axis=0)
test_x_tg = np.mean([p[1] for p in tg_test_parts_all], axis=0)
test_e_tg = np.mean([p[2] for p in tg_test_parts_all], axis=0)

oof_tg_parts  = (oof_l_tg,  oof_x_tg,  oof_e_tg)
test_tg_parts = (test_l_tg, test_x_tg, test_e_tg)

print(f'\nSeed-averaged OOF R²  LGBM={r2_score(y_tg, oof_l_tg):.4f}  '
      f'XGB={r2_score(y_tg, oof_x_tg):.4f}  ET={r2_score(y_tg, oof_e_tg):.4f}')


In [ ]:
print('=' * 60)
print('  Egc  (chain band gap)')
print('=' * 60)

egc_oof_parts_all  = []
egc_test_parts_all = []

for seed in SEEDS:
    print(f'\n  seed={seed}')
    oof_parts, test_parts = train_ensemble(X_egc, y_egc, X_egc_test, 'egc', seed=seed)
    egc_oof_parts_all.append(oof_parts)
    egc_test_parts_all.append(test_parts)

oof_l_egc  = np.mean([p[0] for p in egc_oof_parts_all],  axis=0)
oof_x_egc  = np.mean([p[1] for p in egc_oof_parts_all],  axis=0)
oof_e_egc  = np.mean([p[2] for p in egc_oof_parts_all],  axis=0)
test_l_egc = np.mean([p[0] for p in egc_test_parts_all], axis=0)
test_x_egc = np.mean([p[1] for p in egc_test_parts_all], axis=0)
test_e_egc = np.mean([p[2] for p in egc_test_parts_all], axis=0)

oof_egc_parts  = (oof_l_egc,  oof_x_egc,  oof_e_egc)
test_egc_parts = (test_l_egc, test_x_egc, test_e_egc)

print(f'\nSeed-averaged OOF R²  LGBM={r2_score(y_egc, oof_l_egc):.4f}  '
      f'XGB={r2_score(y_egc, oof_x_egc):.4f}  ET={r2_score(y_egc, oof_e_egc):.4f}')


## Optimise Blend Weights on OOF

Find the weight vector `[w_lgbm, w_xgb, w_cat]` that maximises OOF R² per target via constrained optimisation (`sum=1`, all weights ≥0).  
These weights are then applied to the **test predictions** — not just used diagnostically.  
The individual test arrays (`test_lgbm`, `test_xgb`, `test_cat`) are returned from `train_ensemble` for exactly this purpose.

In [ ]:
from scipy.optimize import minimize

def find_best_weights(oof_parts, y_true):
    oof_l, oof_x, oof_e = oof_parts
    stack = np.column_stack([oof_l, oof_x, oof_e])
    def neg_r2(w):
        return -r2_score(y_true, stack @ w)
    res = minimize(
        neg_r2, x0=[1/3, 1/3, 1/3], method='SLSQP',
        bounds=[(0, 1)] * 3,
        constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}
    )
    return res.x

w_tg  = find_best_weights(oof_tg_parts,  y_tg)
w_egc = find_best_weights(oof_egc_parts, y_egc)

print(f'Optimal Tg  weights — LGBM: {w_tg[0]:.3f}  XGB: {w_tg[1]:.3f}  ET: {w_tg[2]:.3f}')
print(f'Optimal Egc weights — LGBM: {w_egc[0]:.3f}  XGB: {w_egc[1]:.3f}  ET: {w_egc[2]:.3f}')

oof_l_tg,  oof_x_tg,  oof_e_tg  = oof_tg_parts
oof_l_egc, oof_x_egc, oof_e_egc = oof_egc_parts
test_l_tg,  test_x_tg,  test_e_tg  = test_tg_parts
test_l_egc, test_x_egc, test_e_egc = test_egc_parts

oof_tg_opt  = w_tg[0]*oof_l_tg   + w_tg[1]*oof_x_tg   + w_tg[2]*oof_e_tg
oof_egc_opt = w_egc[0]*oof_l_egc  + w_egc[1]*oof_x_egc  + w_egc[2]*oof_e_egc

r2_tg  = r2_score(y_tg,  oof_tg_opt)
r2_egc = r2_score(y_egc, oof_egc_opt)

print(f'\nOOF R² Tg  : {r2_tg:.4f}   (v4: 0.9082)')
print(f'OOF R² Egc : {r2_egc:.4f}   (v4: 0.9151)')
print(f'Mean OOF R²: {(r2_tg + r2_egc)/2:.4f}   (v4: 0.9117)')

pred_tg  = w_tg[0]*test_l_tg   + w_tg[1]*test_x_tg   + w_tg[2]*test_e_tg
pred_egc = w_egc[0]*test_l_egc  + w_egc[1]*test_x_egc  + w_egc[2]*test_e_egc

print('\nTest predictions updated with optimised weights.')


In [ ]:
sub_tg          = test_tg[['id']].copy()
sub_tg['target'] = pred_tg

sub_egc          = test_egc[['id']].copy()
sub_egc['target'] = pred_egc

submission = (
    pd.concat([sub_tg, sub_egc], axis=0)
    .sort_values('id')
    .reset_index(drop=True)
)

assert submission.shape[0] == len(test), 'Row count mismatch!'
assert submission['target'].isna().sum() == 0, 'NaN in predictions!'

submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Submission shape:', submission.shape)
print(submission.head(10))
print('\nsubmission.csv saved.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_tg, oof_tg_opt, alpha=0.25, s=8)
lo, hi = min(y_tg.min(), oof_tg_opt.min()), max(y_tg.max(), oof_tg_opt.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[0].set_xlabel('True Tg (°C)', fontsize=12)
axes[0].set_ylabel('Pred Tg (°C)', fontsize=12)
axes[0].set_title(f'Tg OOF  R² = {r2_tg:.4f}', fontsize=13)
axes[0].grid(alpha=0.3)

axes[1].scatter(y_egc, oof_egc_opt, alpha=0.25, s=8, color='darkorange')
lo, hi = min(y_egc.min(), oof_egc_opt.min()), max(y_egc.max(), oof_egc_opt.max())
axes[1].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[1].set_xlabel('True Egc (eV)', fontsize=12)
axes[1].set_ylabel('Pred Egc (eV)', fontsize=12)
axes[1].set_title(f'Egc OOF  R² = {r2_egc:.4f}', fontsize=13)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'v6 OOF Predicted vs Actual  |  Mean R² = {(r2_tg+r2_egc)/2:.4f}   (v4: 0.896 LB)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/kaggle/working/oof_scatter_v6.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved.')